# GAT-GRU + PPO trong Digital Twin — v1

Notebook này không dùng lookup và không dùng MLP sinh toàn bộ next-state. Digital Twin tách riêng: workload ngoại sinh, lifecycle replica, queue/backlog, CPU theo work conservation, phân phối latency theo từng service và lớp quan sát Jaeger 10%. Test calibration bị khóa cho đến bước đánh giá cuối.


In [ ]:
%pip install -q "gymnasium==1.2.0" "stable-baselines3>=2.4,<3" "scikit-learn>=1.3"


## 1. Cấu hình, contract và tái lập


In [ ]:
import os, sys, json, random, hashlib, math
from dataclasses import dataclass
from pathlib import Path
from collections import defaultdict, deque
import numpy as np
import pandas as pd
import torch
from torch import nn
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

def find_repo():
    configured = os.environ.get('KAGGLE_REPO_ROOT')
    roots = [Path(configured)] if configured else [Path.cwd(), Path('/kaggle/input')]
    for root in roots:
        if root.is_dir() and (root/'modeling-common'/'forecast_data.py').is_file(): return root.resolve()
        if root.is_dir():
            hits = list(root.rglob('modeling-common/forecast_data.py'))
            if len(hits) == 1: return hits[0].parents[1].resolve()
    raise FileNotFoundError('Attach repository dataset or set KAGGLE_REPO_ROOT')

ROOT = find_repo(); MODEL_ROOT = ROOT/'modeling-common'
sys.path.insert(0, str(MODEL_ROOT))
import forecast_data as fd
from gatgru_inference import GATGRUPredictor, load_named_edges

@dataclass(frozen=True)
class Config:
    tick_sec: int = 10
    decision_ticks: int = 3
    history_ticks: int = 18
    cooldown_ticks: int = 6
    sampling: float = 0.10
    slo_ms: float = 300.0
    min_replicas: int = 1
    max_replicas: int = 3
    max_total_app_pods: int = 16
    controlled: tuple = ('frontend','cartservice','currencyservice','productcatalogservice','recommendationservice')
    mdn_components: int = 4
    mdn_epochs: int = 40
    mdn_batch: int = 4096
    reward_sla: float = 5.0
    reward_error: float = 2.0
    reward_pods: float = 0.02
    reward_scale: float = 0.10
    reward_reversal: float = 0.25
    reward_blocked: float = 0.05
    reward_ood: float = 1.0
CFG = Config()
OUT = Path('/kaggle/working/digital_twin_rl'); OUT.mkdir(parents=True, exist_ok=True)
print('repo=', ROOT, 'device=', DEVICE, 'output=', OUT)


## 2. Nạp manifest hiệu chỉnh và khóa test


In [ ]:
PROTOCOL = 'digital_twin_calibration_v1'
def sha256(path):
    h=hashlib.sha256()
    with Path(path).open('rb') as f:
        for chunk in iter(lambda:f.read(1<<20), b''): h.update(chunk)
    return h.hexdigest()

def find_manifest():
    configured=os.environ.get('KAGGLE_TWIN_MANIFEST')
    if configured: return Path(configured).resolve()
    hits=list(Path('/kaggle/input').rglob('digital_twin_manifest.json')) if Path('/kaggle/input').exists() else []
    if len(hits)!=1: raise FileNotFoundError('Set KAGGLE_TWIN_MANIFEST to the unique exported manifest')
    return hits[0].resolve()

MANIFEST_PATH=find_manifest(); DATA_ROOT=MANIFEST_PATH.parent.resolve()
MANIFEST=json.loads(MANIFEST_PATH.read_text())
assert MANIFEST['protocol']==PROTOCOL
assert MANIFEST['plan']['sampling_percentage']==10
assert MANIFEST['plan']['replica_levels']==[1,2,3]
assert tuple(MANIFEST['plan']['controlled_services'])==CFG.controlled
assert MANIFEST['plan']['max_total_app_pods']==CFG.max_total_app_pods
assert MANIFEST['plan']['window_sec']==CFG.tick_sec*CFG.decision_ticks
if MANIFEST['missing_cells']: raise RuntimeError(f"Incomplete campaign: {len(MANIFEST['missing_cells'])} cells missing")

def inside(root, path):
    try: Path(path).resolve().relative_to(Path(root).resolve()); return True
    except ValueError: return False

def runs_for(split, allow_test=False):
    if split=='test' and not allow_test: raise PermissionError('Calibration test is sealed')
    rows=[]
    for entry in MANIFEST['runs']:
        if entry['split']!=split: continue
        folder=(DATA_ROOT/entry['directory']).resolve()
        if not inside(DATA_ROOT,folder): raise ValueError('Manifest path escape')
        for name,digest in entry['raw_sha256'].items():
            path=folder/name
            if not path.is_file() or sha256(path)!=digest: raise ValueError(f'Changed calibration file: {path}')
        rows.append((entry,folder))
    if not rows: raise ValueError(f'No {split} runs')
    return rows
TRAIN_RUNS=runs_for('train'); VAL_RUNS=runs_for('val')
print('train runs=',len(TRAIN_RUNS),'validation runs=',len(VAL_RUNS),'test remains sealed')


## 3. Hiệu chỉnh twin
CPU được fit bằng mô hình không âm `idle × ready + cost/request × throughput`. Capacity lấy từ miền throughput/replica quan sát. Latency được mô hình bằng hỗn hợp LogNormal có điều kiện theo service, request rate, ready replica và queue pressure; mô hình không dự đoán replica hay workload.


In [ ]:
SERVICES=list(fd.SERVICES); SVC={s:i for i,s in enumerate(SERVICES)}; EDGES=list(fd.STATIC_EDGES)
def load_tables(runs):
    ticks=[]; spans=[]; edges=[]; startup=[]
    for entry,folder in runs:
        node=pd.read_csv(folder/'node_metrics.csv')
        svc=pd.read_csv(folder/'service_metrics.csv')
        tick=node.merge(svc,on=['timestamp','service'],how='left')
        tick['request_count']=tick['request_count'].fillna(0)
        tick['error_count']=tick['error_count'].fillna(0)
        tick['rps']=tick['request_count']/(CFG.tick_sec*CFG.sampling)
        tick['run']=entry['directory']; tick['scenario']=entry['scenario']; tick['split']=entry['split']
        ticks.append(tick)
        raw=pd.read_csv(folder/'jaeger_server_spans.csv')
        raw=raw.merge(tick[['timestamp','service','rps','replicas','cpu_cores']],
                      left_on=['bucket_timestamp','service'],right_on=['timestamp','service'],how='inner')
        cluster=tick.groupby('timestamp',as_index=False).replicas.sum().rename(columns={'timestamp':'bucket_timestamp','replicas':'cluster_replicas'})
        raw=raw.merge(cluster,on='bucket_timestamp',how='left')
        raw['run']=entry['directory']; spans.append(raw)
        edge=pd.read_csv(folder/'edge_metrics.csv'); edge['run']=entry['directory']; edges.append(edge)
        for row in entry['capacity_delay_audit']:
            if row['desired']>1 and row['first_observed_available_delay_sec'] is not None:
                startup.append(dict(service=row['service'],desired=row['desired'],delay=row['first_observed_available_delay_sec']))
    return pd.concat(ticks,ignore_index=True),pd.concat(spans,ignore_index=True),pd.concat(edges,ignore_index=True),pd.DataFrame(startup)

train_ticks,train_spans,train_edges,startup_train=load_tables(TRAIN_RUNS)
val_ticks,val_spans,val_edges,startup_val=load_tables(VAL_RUNS)
for frame in (train_ticks,val_ticks): frame['per_replica']=frame['rps']/frame['replicas'].clip(lower=1)

PARAMS={'sampling':CFG.sampling,'tick_sec':CFG.tick_sec,'services':SERVICES}
PARAMS['cluster_replicas_q99']=float(train_ticks.groupby(['run','timestamp']).replicas.sum().quantile(.99))
PARAMS['mu_rps_per_replica']={}; PARAMS['capacity_identified']={}
PARAMS['cpu']={}; PARAMS['error']={}; PARAMS['resources']={}; PARAMS['support']={}; PARAMS['startup_delay']={}
for service in SERVICES:
    rows=train_ticks[train_ticks.service==service].copy()
    if service=='redis-cart':
        PARAMS['mu_rps_per_replica'][service]=1e6; PARAMS['capacity_identified'][service]=True
        PARAMS['cpu'][service]={'idle_per_replica':float(rows.cpu_cores.median()),'core_per_rps':0.0}
        PARAMS['error'][service]={'base':0.0,'overflow_slope':0.0}
        PARAMS['resources'][service]={'mem_per_replica':float((rows.mem_bytes/rows.replicas.clip(lower=1)).median()),'net_rx_per_rps':0.0,'net_tx_per_rps':0.0}
        PARAMS['support'][service]={'rps_q01':0.0,'rps_q99':0.0,'rho_q99':0.0}; PARAMS['startup_delay'][service]=[]
        continue
    positive=rows[rows.per_replica>0].per_replica
    if len(positive)<20: raise RuntimeError(f'Insufficient capacity support for {service}')
    low_cut=positive.quantile(.30); baseline_latency=rows.loc[rows.per_replica<=low_cut,'latency_p95_ms'].dropna()
    base=float(baseline_latency.median()) if len(baseline_latency) else float('nan')
    threshold=max(base*2,base+20) if np.isfinite(base) else float('inf')
    congested=rows[(rows.per_replica>low_cut)&(rows.latency_p95_ms>threshold)].per_replica
    identified=len(congested)>=10
    mu=congested.quantile(.10) if identified else positive.quantile(.95)*1.10
    PARAMS['mu_rps_per_replica'][service]=float(max(1e-3,mu)); PARAMS['capacity_identified'][service]=bool(identified)
    reg=LinearRegression(positive=True,fit_intercept=False).fit(rows[['replicas','rps']],rows.cpu_cores)
    PARAMS['cpu'][service]={'idle_per_replica':float(reg.coef_[0]),'core_per_rps':float(reg.coef_[1])}
    capacity=PARAMS['mu_rps_per_replica'][service]*rows.replicas.clip(lower=1)
    overflow=((rows.rps-capacity).clip(lower=0)/capacity.clip(lower=1e-3)).to_numpy()
    observed_error=(rows.error_count/rows.request_count.clip(lower=1)).fillna(0).to_numpy()
    err_reg=LinearRegression(positive=True,fit_intercept=False).fit(np.c_[np.ones(len(rows)),overflow],observed_error)
    PARAMS['error'][service]={'base':float(err_reg.coef_[0]),'overflow_slope':float(err_reg.coef_[1])}
    PARAMS['resources'][service]={
      'mem_per_replica':float((rows.mem_bytes/rows.replicas.clip(lower=1)).median()),
      'net_rx_per_rps':float((rows.net_rx_bps/rows.rps.clip(lower=.1)).median()),
      'net_tx_per_rps':float((rows.net_tx_bps/rows.rps.clip(lower=.1)).median())}
    PARAMS['support'][service]={'rps_q01':float(rows.rps.quantile(.01)),'rps_q99':float(rows.rps.quantile(.99)),
                                'rho_q99':float((rows.rps/capacity.clip(lower=1e-3)).quantile(.99))}
    delays=startup_train[startup_train.service==service].delay.to_numpy(float)
    PARAMS['startup_delay'][service]=delays.tolist()

PARAMS['fanout']={}
source_rates=train_ticks[['run','timestamp','service','rps']].rename(columns={'service':'source','rps':'source_rps'})
edge=train_edges.merge(source_rates,on=['run','timestamp','source'],how='left')
edge['call_rate']=edge.call_count/(CFG.tick_sec*CFG.sampling)
for source,target in EDGES:
    rows=edge[(edge.source==source)&(edge.target==target)&(edge.source_rps>0)]
    PARAMS['fanout'][f'{source}->{target}']=float((rows.call_rate/rows.source_rps).median()) if len(rows) else 0.0

def add_mdn_features(spans):
    out=spans.copy(); mu=out.service.map(PARAMS['mu_rps_per_replica']).astype(float)
    capacity=mu*out.replicas.clip(lower=1)
    out['rho']=(out.rps/capacity).clip(0,4)
    out['overflow']=((out.rps-capacity).clip(lower=0)/capacity.clip(lower=1e-3)).clip(0,4)
    out['service_id']=out.service.map(SVC)
    out['log_latency']=np.log(out.latency_ms.clip(lower=.01))
    return out[np.isfinite(out[['rps','replicas','rho','overflow','cluster_replicas','log_latency']]).all(axis=1)]
train_mdn=add_mdn_features(train_spans); val_mdn=add_mdn_features(val_spans)
print('ticks=',len(train_ticks),'train spans=',len(train_mdn),'val spans=',len(val_mdn))


In [ ]:
class LatencyMDN(nn.Module):
    def __init__(self,n_services,n_components):
        super().__init__(); self.k=n_components; self.embed=nn.Embedding(n_services,8)
        self.body=nn.Sequential(nn.Linear(13,64),nn.SiLU(),nn.Linear(64,64),nn.SiLU())
        self.head=nn.Linear(64,3*n_components)
    def forward(self,service_id,x):
        raw=self.head(self.body(torch.cat([self.embed(service_id),x],dim=1)))
        logits,loc,scale=raw.chunk(3,dim=1)
        return logits,loc,torch.nn.functional.softplus(scale)+0.03

def tensors(frame):
    x=np.c_[np.log1p(frame.rps),frame.replicas/CFG.max_replicas,frame.rho,frame.overflow,
            frame.cluster_replicas/CFG.max_total_app_pods].astype('float32')
    return (torch.as_tensor(frame.service_id.to_numpy(),dtype=torch.long),
            torch.as_tensor(x),torch.as_tensor(frame.log_latency.to_numpy('float32')))
def mdn_nll(logits,loc,scale,y):
    log_prob=torch.distributions.Normal(loc,scale).log_prob(y[:,None])+torch.log_softmax(logits,dim=1)
    return -torch.logsumexp(log_prob,dim=1).mean()

model=LatencyMDN(len(SERVICES),CFG.mdn_components).to(DEVICE)
optimizer=torch.optim.AdamW(model.parameters(),lr=2e-3,weight_decay=1e-4)
sid,x,y=tensors(train_mdn); dataset=torch.utils.data.TensorDataset(sid,x,y)
loader=torch.utils.data.DataLoader(dataset,batch_size=CFG.mdn_batch,shuffle=True,generator=torch.Generator().manual_seed(SEED))
best=None; best_val=float('inf'); history=[]
vsid,vx,vy=(z.to(DEVICE) for z in tensors(val_mdn))
for epoch in range(CFG.mdn_epochs):
    model.train(); losses=[]
    for bsid,bx,by in loader:
        bsid,bx,by=bsid.to(DEVICE),bx.to(DEVICE),by.to(DEVICE)
        loss=mdn_nll(*model(bsid,bx),by); optimizer.zero_grad(); loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(),5); optimizer.step(); losses.append(loss.item())
    model.eval()
    with torch.no_grad(): val=float(mdn_nll(*model(vsid,vx),vy).cpu())
    history.append({'epoch':epoch,'train_nll':float(np.mean(losses)),'val_nll':val})
    if val<best_val: best_val=val; best={k:v.detach().cpu().clone() for k,v in model.state_dict().items()}
model.load_state_dict(best); model.eval(); pd.DataFrame(history).to_csv(OUT/'mdn_history.csv',index=False)
torch.save({'state_dict':best,'params':PARAMS,'config':CFG.__dict__},OUT/'digital_twin.pt')
(OUT/'digital_twin_params.json').write_text(json.dumps(PARAMS,indent=2))
print('best validation NLL=',best_val)


## 4. Validation bắt buộc trước PPO


In [ ]:
@torch.no_grad()
def sample_mdn(service_ids,x,n=64,rng=None):
    rng=np.random.default_rng(SEED) if rng is None else rng
    sid=torch.as_tensor(service_ids,dtype=torch.long,device=DEVICE)
    xx=torch.as_tensor(x,dtype=torch.float32,device=DEVICE)
    logits,loc,scale=model(sid,xx); prob=torch.softmax(logits,1).cpu().numpy()
    loc=loc.cpu().numpy(); scale=scale.cpu().numpy(); out=np.empty((len(x),n))
    for i in range(len(x)):
        component=rng.choice(model.k,size=n,p=prob[i])
        out[i]=np.exp(rng.normal(loc[i,component],scale[i,component]))
    return out

vsid_cpu,vx_cpu,vy_cpu=tensors(val_mdn); draws=sample_mdn(vsid_cpu.numpy(),vx_cpu.numpy(),128)
truth=np.exp(vy_cpu.numpy()); median=np.median(draws,axis=1); lo=np.quantile(draws,.05,axis=1); hi=np.quantile(draws,.95,axis=1)
train_medians=train_mdn.assign(latency=np.exp(train_mdn.log_latency)).groupby('service_id').latency.median()
baseline=val_mdn.service_id.map(train_medians).to_numpy(float)
diag=pd.DataFrame({'service':val_mdn.service.to_numpy(),'truth':truth,'median':median,'lo':lo,'hi':hi,'baseline':baseline})
diag['covered']=(diag.truth>=diag.lo)&(diag.truth<=diag.hi); diag['abs_error']=(diag.truth-diag['median']).abs(); diag['baseline_abs_error']=(diag.truth-diag.baseline).abs()
service_metrics=diag.groupby('service').agg(samples=('truth','size'),coverage90=('covered','mean'),mae_ms=('abs_error','mean'),baseline_mae_ms=('baseline_abs_error','mean')).reset_index()
service_metrics['improvement']=1-service_metrics.mae_ms/service_metrics.baseline_mae_ms.clip(lower=1e-6)
service_metrics.to_csv(OUT/'latency_validation_by_service.csv',index=False)
coverage=float(diag.covered.mean()); mae=float(diag.abs_error.mean()); base_mae=float(diag.baseline_abs_error.mean())
frontend=diag[diag.service=='frontend']; slo_rate_error=abs(float((draws[val_mdn.service.to_numpy()=='frontend']>CFG.slo_ms).mean())-float((frontend.truth>CFG.slo_ms).mean()))
level_support=lambda frame: all(set(frame[frame.service==s].replicas.astype(int))>=set(range(1,CFG.max_replicas+1)) for s in CFG.controlled)
startup_services=set(startup_train.service)&set(CFG.controlled)
adj=np.zeros((len(SERVICES),len(SERVICES)),int); fan=np.zeros_like(adj,dtype=float)
for a,b in EDGES: adj[SVC[a],SVC[b]]=1; fan[SVC[a],SVC[b]]=PARAMS['fanout'][f'{a}->{b}']
degree=adj.sum(0); queue=deque(np.flatnonzero(degree==0).tolist()); topo=[]
while queue:
    i=queue.popleft(); topo.append(i)
    for j in np.flatnonzero(adj[i]):
        degree[j]-=1
        if degree[j]==0: queue.append(int(j))
if len(topo)!=len(SERVICES): raise RuntimeError('Static service graph contains a cycle')
def graph_rates(frontend_rps):
    out=np.zeros(len(SERVICES)); out[SVC['frontend']]=frontend_rps
    for i in topo:
        for j in np.flatnonzero(adj[i]): out[j]+=out[i]*fan[i,j]
    return out
observed=val_ticks.pivot_table(index=['run','timestamp'],columns='service',values='rps').reindex(columns=SERVICES).dropna()
predicted=np.stack([graph_rates(value) for value in observed.frontend.to_numpy()])
graph_load_metrics=pd.DataFrame({'service':SERVICES,'normalized_mae':[float(np.mean(np.abs(predicted[:,i]-observed.iloc[:,i].to_numpy())/(observed.iloc[:,i].to_numpy()+1))) for i in range(len(SERVICES))]})
graph_load_metrics.to_csv(OUT/'graph_load_validation.csv',index=False)
GATES={
 'replica_support':level_support(train_ticks) and level_support(val_ticks),
 'startup_support':set(CFG.controlled)<=startup_services,
 'capacity_onset_observed':all(PARAMS['capacity_identified'][s] for s in CFG.controlled),
 'graph_fanout_support':all(PARAMS['fanout'][f'{a}->{b}']>0 for a,b in EDGES if b!='redis-cart'),
 'workload_support_identified':all(PARAMS['support'][s]['rps_q99']>PARAMS['support'][s]['rps_q01'] for s in CFG.controlled),
 'cluster_interaction_support':PARAMS['cluster_replicas_q99']>=len(SERVICES)+2,
 'graph_load_accuracy':all(row.normalized_mae<=.50 for row in graph_load_metrics.itertuples() if row.service in CFG.controlled),
 'finite_nll':np.isfinite(best_val),
 'interval_coverage':all(0.70<=row.coverage90<=0.99 for row in service_metrics.itertuples() if row.service in CFG.controlled),
 'latency_skill':all(row.improvement>0 for row in service_metrics.itertuples() if row.service in CFG.controlled),
 'frontend_slo_calibration':slo_rate_error<=0.10}
pd.Series(GATES,name='passed').to_csv(OUT/'digital_twin_gates.csv')
print({'coverage90':coverage,'mae_ms':mae,'baseline_mae_ms':base_mae,'frontend_slo_rate_error':slo_rate_error}); display(service_metrics); display(graph_load_metrics); print(pd.Series(GATES))
AUTO_GATES_PASS=all(GATES.values())
TWIN_APPROVED=False  # chỉ đổi True sau khi review plots, per-service metrics và held-out rollouts
print('AUTO_GATES_PASS=',AUTO_GATES_PASS,'TWIN_APPROVED=',TWIN_APPROVED)


## 5. Digital Twin Gymnasium: capacity → queue → performance → Jaeger observation


In [ ]:
import gymnasium as gym
from gymnasium import spaces

def episode_rates(runs):
    episodes=[]
    for entry,folder in runs:
        rate=pd.read_csv(folder/'designed_rate.csv')
        series=rate.groupby((rate.t//CFG.tick_sec).astype(int)).expected_rps.mean().to_numpy(float)
        episodes.append({'key':entry['directory'],'scenario':entry['scenario'],'rates':series})
    return episodes
TRAIN_EPISODES=episode_rates(TRAIN_RUNS); VAL_EPISODES=episode_rates(VAL_RUNS)

class ReplicaLifecycle:
    """Kubernetes control-plane dynamics; contains no learned performance model."""
    def __init__(self,rng): self.rng=rng; self.reset()
    def reset(self):
        n=len(SERVICES); self.desired=np.ones(n,int); self.ready=np.ones(n,int)
        self.pending=[[] for _ in SERVICES]; self.cooldown=np.zeros(n,int); self.last_delta=np.zeros(n,int)
    def apply(self,action):
        result={'changes':0,'reversals':0,'blocked':0}
        for service,delta in zip(CFG.controlled,np.asarray(action,dtype=int)-1):
            i=SVC[service]; delta=int(delta)
            if delta==0: continue
            if self.cooldown[i]>0: result['blocked']+=1; continue
            target=int(np.clip(self.desired[i]+delta,CFG.min_replicas,CFG.max_replicas))
            if target==self.desired[i]: result['blocked']+=1; continue
            if target>self.desired[i] and self.desired.sum()+target-self.desired[i]>CFG.max_total_app_pods:
                result['blocked']+=1; continue
            if self.last_delta[i]*delta<0: result['reversals']+=1
            self.last_delta[i]=delta; result['changes']+=1
            self.desired[i]=target; self.cooldown[i]=CFG.cooldown_ticks
            if target<self.ready[i]: self.ready[i]=target; self.pending[i]=[]
            else:
                while self.ready[i]+len(self.pending[i])>target: self.pending[i].pop()
                while self.ready[i]+len(self.pending[i])<target:
                    samples=PARAMS['startup_delay'].get(service) or [90.0]
                    self.pending[i].append(float(self.rng.choice(samples)))
        return result
    def advance(self):
        self.cooldown=np.maximum(0,self.cooldown-1)
        for i in range(len(SERVICES)):
            self.pending[i]=[delay-CFG.tick_sec for delay in self.pending[i]]
            self.ready[i]+=sum(delay<=0 for delay in self.pending[i])
            self.pending[i]=[delay for delay in self.pending[i] if delay>0]

class GraphLoadModel:
    """Static service topology plus learned, dynamic caller-to-callee fan-out."""
    def __init__(self):
        n=len(SERVICES); self.adjacency=np.zeros((n,n),dtype=np.float32); self.fanout=np.zeros((n,n),dtype=np.float32)
        for source,target in EDGES:
            i,j=SVC[source],SVC[target]; self.adjacency[i,j]=1
            self.fanout[i,j]=max(0,float(PARAMS['fanout'][f'{source}->{target}']))
        indegree=self.adjacency.sum(axis=0).astype(int); queue=deque(np.flatnonzero(indegree==0).tolist()); order=[]
        while queue:
            i=queue.popleft(); order.append(i)
            for j in np.flatnonzero(self.adjacency[i]):
                indegree[j]-=1
                if indegree[j]==0: queue.append(int(j))
        if len(order)!=n: raise RuntimeError('Static microservice graph must be acyclic for load propagation')
        self.order=order
    def propagate(self,offered_rps):
        rps=np.zeros(len(SERVICES),dtype=float); rps[SVC['frontend']]=max(0,float(offered_rps))
        for i in self.order:
            for j in np.flatnonzero(self.adjacency[i]): rps[j]+=rps[i]*self.fanout[i,j]
        return rps

class InfrastructureDynamics:
    """Analytical capacity/queue state calibrated from profiling traces."""
    def reset(self): self.backlog=np.zeros(len(SERVICES),dtype=float)
    def step(self,rps,ready):
        mu=np.array([PARAMS['mu_rps_per_replica'][s] for s in SERVICES]); capacity=mu*np.maximum(ready,1)
        demand=rps+self.backlog/CFG.tick_sec; completed=np.minimum(demand,capacity)
        self.backlog=np.maximum(0,self.backlog+(rps-completed)*CFG.tick_sec)
        rho=np.clip(demand/np.maximum(capacity,1e-3),0,4); overflow=np.clip(self.backlog/np.maximum(capacity*CFG.tick_sec,1e-3),0,4)
        cpu=np.array([PARAMS['cpu'][s]['idle_per_replica']*ready[i]+PARAMS['cpu'][s]['core_per_rps']*completed[i] for i,s in enumerate(SERVICES)])
        error=np.array([np.clip(PARAMS['error'][s]['base']+PARAMS['error'][s]['overflow_slope']*overflow[i],0,1) for i,s in enumerate(SERVICES)])
        mem=np.array([PARAMS['resources'][s]['mem_per_replica']*ready[i]/2**20 for i,s in enumerate(SERVICES)])
        ood=np.zeros(len(SERVICES),dtype=float)
        for i,s in enumerate(SERVICES):
            support=PARAMS['support'][s]; rps_hi=max(.1,support['rps_q99']); rho_hi=max(.1,support['rho_q99'])
            ood[i]=max(0,(rps[i]-rps_hi)/rps_hi,(rho[i]-rho_hi)/rho_hi)
        cluster_hi=max(1,PARAMS['cluster_replicas_q99'])
        ood=np.maximum(ood,max(0,(ready.sum()-cluster_hi)/cluster_hi))
        return {'completed':completed,'rho':rho,'overflow':overflow,'cpu':cpu,'error':error,'mem_mib':mem,'ood':ood}

class SurrogateObservationModel:
    """Learned stochastic latency model and Jaeger-compatible graph observation."""
    def __init__(self,rng,graph): self.rng=rng; self.graph=graph; self.prev_rps=np.zeros(len(SERVICES))
    def reset(self): self.prev_rps=np.zeros(len(SERVICES))
    def build(self,rps,ready,dyn):
        feat=np.c_[np.log1p(rps),ready/CFG.max_replicas,dyn['rho'],dyn['overflow'],
                   np.full(len(SERVICES),ready.sum()/CFG.max_total_app_pods)].astype('float32')
        draws=sample_mdn(np.arange(len(SERVICES)),feat,512,self.rng); p50=np.zeros(len(SERVICES)); p95=np.zeros(len(SERVICES))
        for i,service in enumerate(SERVICES):
            if service=='redis-cart': continue
            population=max(1,int(round(rps[i]*CFG.tick_sec)))
            observed=max(1,int(self.rng.binomial(population,CFG.sampling)))
            sample=draws[i,:min(observed,draws.shape[1])]; p50[i]=np.quantile(sample,.50); p95[i]=np.quantile(sample,.95)
        node={name:np.zeros(len(SERVICES),dtype=np.float32) for name in fd.NF}
        values={'rps_in':rps,'rps_in_delta':rps-self.prev_rps,'rps_per_replica':rps/np.maximum(ready,1),
                'latency_p50_ms':p50,'latency_p95_ms':p95,'error_rate':dyn['error'],'cpu_cores':dyn['cpu'],
                'mem_mib':dyn['mem_mib'],'replicas':ready}
        for name,value in values.items(): node[name]=np.asarray(value,dtype=np.float32)
        for i,s in enumerate(SERVICES):
            resource=PARAMS['resources'][s]; node['net_rx_kBps'][i]=resource['net_rx_per_rps']*rps[i]/1024
            node['net_tx_kBps'][i]=resource['net_tx_per_rps']*rps[i]/1024
        edge={name:np.zeros(len(EDGES),dtype=np.float32) for name in fd.EF}
        for k,(source,target) in enumerate(EDGES):
            i,j=SVC[source],SVC[target]; edge['call_ratio'][k]=self.graph.fanout[i,j]
            edge['call_rate'][k]=rps[i]*self.graph.fanout[i,j]; edge['edge_latency_p95_ms'][k]=p95[j]
        self.prev_rps=rps.copy()
        return np.stack([node[name] for name in fd.NF],axis=-1),np.stack([edge[name] for name in fd.EF],axis=-1),p50,p95

class DigitalTwin:
    """Thin orchestrator: lifecycle -> graph load -> infrastructure -> learned observation."""
    def __init__(self,seed=0):
        self.rng=np.random.default_rng(seed); self.graph=GraphLoadModel(); self.lifecycle=ReplicaLifecycle(self.rng)
        self.infrastructure=InfrastructureDynamics(); self.surrogate=SurrogateObservationModel(self.rng,self.graph); self.reset()
    def _aliases(self):
        self.desired=self.lifecycle.desired; self.ready=self.lifecycle.ready; self.pending=self.lifecycle.pending
        self.cooldown=self.lifecycle.cooldown; self.backlog=self.infrastructure.backlog
    def reset(self):
        self.lifecycle.reset(); self.infrastructure.reset(); self.surrogate.reset(); self.history=deque(maxlen=CFG.history_ticks); self._aliases(); return self
    def apply_action(self,action): return self.lifecycle.apply(action)
    def tick(self,offered_rps):
        self.lifecycle.advance(); rps=self.graph.propagate(offered_rps)
        dyn=self.infrastructure.step(rps,self.lifecycle.ready); x,e,p50,p95=self.surrogate.build(rps,self.lifecycle.ready,dyn)
        self.history.append((x,e)); self._aliases()
        return {'x':x,'e':e,'adjacency':self.graph.adjacency.copy(),'rps':rps,'cpu':dyn['cpu'],'mem_mib':dyn['mem_mib'],
                'p50':p50,'p95':p95,'error':dyn['error'],'ood':dyn['ood'],'ready':self.ready.copy(),
                'pending':np.array([len(v) for v in self.pending]),'cooldown':self.cooldown.copy(),'backlog':self.backlog.copy()}

def find_checkpoint():
    configured=os.environ.get('KAGGLE_GATGRU_CHECKPOINT')
    if configured and Path(configured).is_file(): return Path(configured)
    hits=list(Path('/kaggle/input').rglob('gatgru_selected_model_seed_4.pt')) if Path('/kaggle/input').exists() else []
    if len(hits)!=1: raise FileNotFoundError('Attach frozen seed-4 checkpoint or set KAGGLE_GATGRU_CHECKPOINT')
    return hits[0]
PREDICTOR=GATGRUPredictor(find_checkpoint(),load_named_edges(ROOT/'configs/gatgru/graph_metadata.json'),DEVICE)

class DigitalTwinEnv(gym.Env):
    metadata={'render_modes':[]}
    def __init__(self,episodes,seed=0,sequential=False):
        self.episodes=episodes; self.rng=np.random.default_rng(seed); self.twin=DigitalTwin(seed); self.sequential=sequential; self.episode_cursor=0
        self.action_space=spaces.MultiDiscrete(np.full(len(CFG.controlled),3))
        self.current_fields=('rps','cpu','mem_mib','p95','ready','pending','cooldown','backlog')
        self.obs_dim=len(CFG.controlled)*len(self.current_fields)+2*len(CFG.controlled)*2+len(EDGES)*len(PREDICTOR.meta['edge_features'])
        self.observation_space=spaces.Box(-np.inf,np.inf,shape=(self.obs_dim,),dtype=np.float32)
    def _forecast(self):
        if len(self.twin.history)<CFG.history_ticks: return np.zeros((6,len(SERVICES),2),np.float32)
        # Select by name; never assume selected features are a prefix of the full contract.
        xs=np.stack([np.stack([row[0][:,fd.NF.index(name)] for name in PREDICTOR.meta['node_features']],axis=-1) for row in self.twin.history])
        es=np.stack([np.stack([row[1][:,fd.EF.index(name)] for name in PREDICTOR.meta['edge_features']],axis=-1) for row in self.twin.history])
        return PREDICTOR.predict(xs[None],es[None])[0]
    def _obs(self,state):
        ids=[SVC[s] for s in CFG.controlled]; current=np.stack([state[k][ids] for k in self.current_fields],axis=1).ravel()
        forecast=self._forecast(); target={name:i for i,name in enumerate(PREDICTOR.meta['targets'])}
        future=np.stack([forecast[h,ids][:,[target['rps_in'],target['cpu_cores']]].ravel() for h in (2,5)]).ravel()
        graph=state['e'][:,[fd.EF.index(name) for name in PREDICTOR.meta['edge_features']]].ravel()
        return np.nan_to_num(np.r_[current,future,graph],nan=0,posinf=1e6,neginf=-1e6).astype(np.float32)
    def reset(self,seed=None,options=None):
        super().reset(seed=seed)
        if self.sequential: self.ep=self.episodes[self.episode_cursor%len(self.episodes)]; self.episode_cursor+=1
        else: self.ep=self.episodes[int(self.rng.integers(len(self.episodes)))]
        self.twin.reset(); self.t=0
        state=None
        for _ in range(CFG.history_ticks): state=self.twin.tick(self.ep['rates'][self.t]); self.t+=1
        return self._obs(state),{'episode':self.ep['key']}
    def step(self,action):
        action_result=self.twin.apply_action(action); reward=0; state=None
        for _ in range(CFG.decision_ticks):
            state=self.twin.tick(self.ep['rates'][self.t]); self.t+=1
            frontend=state['p95'][SVC['frontend']]; excess=max(0,frontend/CFG.slo_ms-1)
            reward-=CFG.reward_sla*excess**2+CFG.reward_error*state['error'][SVC['frontend']]+CFG.reward_pods*np.sum(state['ready'])
            reward-=CFG.reward_ood*np.mean(state['ood'][[SVC[s] for s in CFG.controlled]])
        reward-=CFG.reward_scale*action_result['changes']+CFG.reward_reversal*action_result['reversals']+CFG.reward_blocked*action_result['blocked']
        terminated=self.t>=len(self.ep['rates']); truncated=False
        info={'frontend_p95_ms':float(state['p95'][SVC['frontend']]),'frontend_error_rate':float(state['error'][SVC['frontend']]),
              'total_replicas':int(state['ready'].sum()),'ood_score':float(np.mean(state['ood'])),**action_result}
        return self._obs(state),float(reward),terminated,truncated,info


## 6. Sanity check và PPO multi-seed


In [ ]:
from stable_baselines3.common.env_checker import check_env
env=DigitalTwinEnv(TRAIN_EPISODES,SEED); check_env(env,warn=True)
obs,info=env.reset(); done=False; steps=0; finite=True
while not done and steps<20:
    obs,reward,terminated,truncated,info=env.step(env.action_space.sample()); finite&=np.isfinite(obs).all() and np.isfinite(reward); done=terminated or truncated; steps+=1
probe_load=max(np.quantile(ep['rates'],.90) for ep in VAL_EPISODES); action_effect=[]
for seed in range(16):
    low=DigitalTwin(seed); high=DigitalTwin(seed); high.ready[SVC['frontend']]=high.desired[SVC['frontend']]=3
    low_p=[]; high_p=[]
    for _ in range(6): low_p.append(low.tick(probe_load)['p95'][SVC['frontend']]); high_p.append(high.tick(probe_load)['p95'][SVC['frontend']])
    action_effect.append(np.mean(low_p)-np.mean(high_p))

def conditional_rollout(entry,folder,seeds=range(4)):
    # Validation-only diagnostic. Observed ready replicas are an explicit oracle schedule,
    # never an autonomous PPO rollout and never read from test.
    designed=pd.read_csv(folder/'designed_rate.csv')
    rate=designed.groupby((designed.t//CFG.tick_sec).astype(int)).expected_rps.mean().to_numpy(float)
    node=pd.read_csv(folder/'node_metrics.csv'); ready=node.pivot(index='timestamp',columns='service',values='replicas').reindex(columns=SERVICES)
    actual=pd.read_csv(folder/'service_metrics.csv').query("service=='frontend'").set_index('timestamp').latency_p95_ms
    start=int(json.loads((folder/'collect_report.json').read_text())['start'])
    times=[int(ts) for ts in ready.index.intersection(actual.index) if 0<=int((ts-start)//CFG.tick_sec)<len(rate)]
    if len(times)<10: raise RuntimeError(f'Insufficient conditional rollout windows: {entry["directory"]}')
    predictions=[]
    for seed in seeds:
        twin=DigitalTwin(50_000+seed); values=[]
        for ts in times:
            observed=ready.loc[ts].to_numpy(int); twin.lifecycle.ready[:]=observed; twin.lifecycle.desired[:]=observed
            values.append(twin.tick(rate[int((ts-start)//CFG.tick_sec)])['p95'][SVC['frontend']])
        predictions.append(values)
    truth=actual.loc[times].to_numpy(float); pred=np.mean(predictions,axis=0)
    baseline=np.full(len(truth),train_ticks.query("service=='frontend'").latency_p95_ms.median())
    mask=np.isfinite(truth)&np.isfinite(pred); truth,pred,baseline=truth[mask],pred[mask],baseline[mask]
    return {'scenario':entry['scenario'],'run':entry['directory'],'samples':len(truth),
            'mae_ms':float(mean_absolute_error(truth,pred)),'baseline_mae_ms':float(mean_absolute_error(truth,baseline)),
            'slo_rate_error':abs(float(np.mean(pred>CFG.slo_ms))-float(np.mean(truth>CFG.slo_ms)))}

conditional=pd.DataFrame([conditional_rollout(entry,folder) for entry,folder in VAL_RUNS])
conditional['improvement']=1-conditional.mae_ms/conditional.baseline_mae_ms.clip(lower=1e-6)
conditional.to_csv(OUT/'conditional_validation_rollout.csv',index=False)
scenario_rollout=conditional.groupby('scenario').agg(samples=('samples','sum'),mae_ms=('mae_ms','mean'),baseline_mae_ms=('baseline_mae_ms','mean'),slo_rate_error=('slo_rate_error','mean')).reset_index()
scenario_rollout['improvement']=1-scenario_rollout.mae_ms/scenario_rollout.baseline_mae_ms.clip(lower=1e-6)
GATES['conditional_rollout_skill']=bool((scenario_rollout.improvement>0).all())
GATES['conditional_rollout_slo_calibration']=bool((scenario_rollout.slo_rate_error<=.15).all())
GATES['autonomous_finite']=bool(finite); GATES['replica_action_sensitivity']=float(np.median(action_effect))>0
AUTO_GATES_PASS=all(GATES.values()); pd.Series(GATES,name='passed').to_csv(OUT/'digital_twin_gates.csv')
print('sanity steps=',steps,'median frontend benefit from 3 vs 1 replicas (ms)=',np.median(action_effect)); display(scenario_rollout); print(pd.Series(GATES))
if not AUTO_GATES_PASS: raise RuntimeError('Automatic Digital Twin gates failed; PPO is blocked')
if not TWIN_APPROVED: raise RuntimeError('Review validation outputs, then explicitly set TWIN_APPROVED=True')


In [ ]:
from stable_baselines3 import PPO
from stable_baselines3.common.vec_env import DummyVecEnv, VecNormalize
from stable_baselines3.common.callbacks import EvalCallback, BaseCallback
class SaveBestNorm(BaseCallback):
    def __init__(self,path): super().__init__(); self.path=Path(path)
    def _on_step(self):
        norm=self.model.get_vec_normalize_env()
        if norm is None: raise RuntimeError('VecNormalize is missing')
        norm.save(self.path); return True
RUN_MULTI_SEED=True; TOTAL_TIMESTEPS=300_000; PPO_SEEDS=(0,1,2,3,4) if RUN_MULTI_SEED else (SEED,)
for seed in PPO_SEEDS:
    train=VecNormalize(DummyVecEnv([lambda seed=seed:DigitalTwinEnv(TRAIN_EPISODES,seed)]),norm_obs=True,norm_reward=True,clip_obs=10)
    val=VecNormalize(DummyVecEnv([lambda seed=seed:DigitalTwinEnv(VAL_EPISODES,10_000+seed,sequential=True)]),training=False,norm_obs=True,norm_reward=False,clip_obs=10)
    model_ppo=PPO('MlpPolicy',train,seed=seed,n_steps=2048,batch_size=256,learning_rate=3e-4,gamma=.99,gae_lambda=.95,clip_range=.2,ent_coef=.01,verbose=1,device=DEVICE)
    seed_dir=OUT/f'ppo_seed_{seed}'; seed_dir.mkdir(exist_ok=True)
    callback=EvalCallback(val,best_model_save_path=str(seed_dir),log_path=str(seed_dir),eval_freq=10_000,n_eval_episodes=len(VAL_EPISODES),deterministic=True,callback_on_new_best=SaveBestNorm(seed_dir/'best_vecnormalize.pkl'))
    model_ppo.learn(TOTAL_TIMESTEPS,callback=callback)
    train.save(seed_dir/'final_vecnormalize.pkl')
print('Training complete. Evaluate every selected best_model.zip on sealed test exactly once.')


In [ ]:
# Final evaluation: this is the only cell allowed to unseal calibration test.
def evaluate_best(seed,episodes,split):
    rows=[]; seed_dir=OUT/f'ppo_seed_{seed}'
    for episode in episodes:
        base=DummyVecEnv([lambda episode=episode,seed=seed:DigitalTwinEnv([episode],20_000+seed)])
        vec=VecNormalize.load(seed_dir/'best_vecnormalize.pkl',base); vec.training=False; vec.norm_reward=False
        agent=PPO.load(seed_dir/'best_model.zip',env=vec,device=DEVICE)
        obs=vec.reset(); done=np.array([False]); rewards=[]; p95=[]; replicas=[]; changes=[]; reversals=[]; ood=[]
        while not bool(done[0]):
            action,_=agent.predict(obs,deterministic=True); obs,reward,done,infos=vec.step(action)
            rewards.append(float(reward[0])); p95.append(infos[0]['frontend_p95_ms'])
            replicas.append(infos[0]['total_replicas']); changes.append(infos[0]['changes'])
            reversals.append(infos[0]['reversals']); ood.append(infos[0]['ood_score'])
        rows.append({'seed':seed,'split':split,'episode':episode['key'],'scenario':episode['scenario'],
                     'return':sum(rewards),'mean_p95_ms':float(np.mean(p95)),
                     'slo_violation_rate':float(np.mean(np.asarray(p95)>CFG.slo_ms)),
                     'mean_total_replicas':float(np.mean(replicas)),'scaling_events':int(np.sum(changes)),
                     'action_reversals':int(np.sum(reversals)),'mean_ood_score':float(np.mean(ood))})
    return rows
validation_rows=[]
for seed in PPO_SEEDS: validation_rows.extend(evaluate_best(seed,VAL_EPISODES,'validation'))
pd.DataFrame(validation_rows).to_csv(OUT/'ppo_validation_by_episode.csv',index=False)
TEST_RUNS=runs_for('test',allow_test=True); TEST_EPISODES=episode_rates(TEST_RUNS)
test_rows=[]
for seed in PPO_SEEDS: test_rows.extend(evaluate_best(seed,TEST_EPISODES,'test'))
test_df=pd.DataFrame(test_rows); test_df.to_csv(OUT/'ppo_test_by_episode.csv',index=False)
summary=test_df.groupby('seed')[['return','mean_p95_ms','slo_violation_rate','mean_total_replicas','scaling_events','action_reversals','mean_ood_score']].mean()
summary.agg(['mean','std']).T.to_csv(OUT/'ppo_test_mean_std.csv'); display(summary.agg(['mean','std']).T)


## Điều kiện báo cáo
Không dùng kết quả nếu Digital Twin chưa đạt gate theo từng service/scenario, policy đi ngoài replica 1–3, hoặc best checkpoint không được ghép đúng `VecNormalize`. Test calibration chỉ được mở sau khi model/twin/reward và checkpoint-selection protocol đã đóng băng.
